# Round 2 Result Pipeline

This notebook calls `scripts/generate_results.py` so the notebook and command-line workflow stay aligned.

In [ ]:
# Run once per fresh environment.
%pip install -q numpy pandas matplotlib

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts' / 'generate_results.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
print(PROJECT_ROOT)

In [ ]:
# Main knobs. Use PRESET='smoke' for quick checks, 'serious' for local runs,
# and 'server' when running on a compute server.
PRESET = 'smoke'
SECTIONS = 'all'  # step7,sweeps,regimes,dp,gh or all
OUTPUT_DIR = 'results/round2_notebook_run'

# Override with integers or keep None to use the preset defaults.
EPISODES = None
VOI_SAMPLES = None
BLINKERED_SAMPLES = None
COMMON_OBSERVATIONS = 'auto'  # auto, on, off
OBSERVATIONS_PER_PERSON = None

# Model/evaluation precision knobs.
ALLOCATION_GRID_SIZE = None
EXPECTED_UTILITY_DRAWS = None
TERMINAL_INTEGRATION = None  # None, 'monte_carlo', or 'gauss_hermite'
GAUSS_HERMITE_ORDER = 15

# Limit these for smoke tests; remove limits for fuller sweeps.
ENVIRONMENTS = []  # e.g., ['baseline', 'scarce_time']
SWEEP_FEATURES = []  # e.g., ['total_time', 'mu_need']
MAX_SWEEP_VALUES_PER_FEATURE = None

# DP diagnostic knobs.
DP_MAX_SAMPLES_VALUES = '2,4,6,10'
DP_MEAN_GRID_SIZES = '7,11,21,50'
DP_OBSERVATION_BRANCHES = '3,5'

RUN_NOW = True

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'generate_results.py'),
    '--preset', PRESET,
    '--sections', SECTIONS,
    '--output-dir', OUTPUT_DIR,
    '--common-observations', COMMON_OBSERVATIONS,
    '--gauss-hermite-order', str(GAUSS_HERMITE_ORDER),
    '--dp-max-samples-values', DP_MAX_SAMPLES_VALUES,
    '--dp-mean-grid-sizes', DP_MEAN_GRID_SIZES,
    '--dp-observation-branches', DP_OBSERVATION_BRANCHES,
]

optional_args = {
    '--episodes': EPISODES,
    '--voi-samples': VOI_SAMPLES,
    '--blinkered-samples': BLINKERED_SAMPLES,
    '--observations-per-person': OBSERVATIONS_PER_PERSON,
    '--allocation-grid-size': ALLOCATION_GRID_SIZE,
    '--expected-utility-draws': EXPECTED_UTILITY_DRAWS,
    '--terminal-integration': TERMINAL_INTEGRATION,
    '--max-sweep-values-per-feature': MAX_SWEEP_VALUES_PER_FEATURE,
}
for flag, value in optional_args.items():
    if value is not None:
        cmd.extend([flag, str(value)])
for environment in ENVIRONMENTS:
    cmd.extend(['--environment', environment])
for feature in SWEEP_FEATURES:
    cmd.extend(['--sweep-feature', feature])

print(' '.join(cmd))
if RUN_NOW:
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
output_path = PROJECT_ROOT / OUTPUT_DIR
print(output_path)
for path in sorted(output_path.rglob('*')):
    if path.is_file():
        print(path.relative_to(output_path))

In [ ]:
import pandas as pd

summary_path = output_path / 'summary.md'
if summary_path.exists():
    print(summary_path.read_text()[:2000])

csv_path = output_path / 'step7_final_choice_comparison.csv'
if csv_path.exists():
    display(pd.read_csv(csv_path).head())